# DarkLens: fine-tune DistilBERT (Colab)

**How to run**
1. Open https://colab.research.google.com > *Upload* > choose this notebook.
2. *Runtime > Change runtime type > T4 GPU*.
3. In the left *Files* panel upload `binary_train.csv`, `binary_val.csv`, `binary_test.csv` from your laptop's `ml/data/` (create them first with `python -m ml.prepare_data`).
4. *Runtime > Run all* (about 2-3 minutes on a T4).
5. Download `distilbert.zip` (last cell), unzip it into **`ml/models/distilbert/`** on your laptop (so that `ml/models/distilbert/config.json` exists).
6. Run `python -m ml.evaluate` locally to add it to the comparison.

The same code is in `ml/train_distilbert.py`, so you can also train locally.

In [ ]:
!pip -q install transformers accelerate scikit-learn pandas

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, Trainer,
                          TrainingArguments, set_seed)
set_seed(42)
BASE, MAX_LEN = "distilbert-base-uncased", 64
train_df, val_df, test_df = (pd.read_csv(f"binary_{s}.csv", keep_default_na=False) for s in ("train", "val", "test"))
print(len(train_df), len(val_df), len(test_df), "GPU:", torch.cuda.is_available())

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok(list(df["text"]), truncation=True, max_length=MAX_LEN, padding="max_length")
        self.labels = list(df["label"].astype(int))
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i]); return item

def metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"accuracy": accuracy_score(p.label_ids, preds), "f1": f1_score(p.label_ids, preds)}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=2, id2label={0: "not_dark", 1: "dark"}, label2id={"not_dark": 0, "dark": 1})
args = TrainingArguments(output_dir="ckpt", num_train_epochs=3, learning_rate=5e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="no", logging_steps=50, seed=42, report_to=[])
trainer = Trainer(model=model, args=args, train_dataset=TextDataset(train_df),
                  eval_dataset=TextDataset(val_df), compute_metrics=metrics)
trainer.train()

In [ ]:
pred = trainer.predict(TextDataset(test_df))
print(classification_report(test_df["label"], pred.predictions.argmax(-1), target_names=["not_dark", "dark"]))

In [ ]:
trainer.save_model("distilbert"); tok.save_pretrained("distilbert")
!zip -qr distilbert.zip distilbert
from google.colab import files; files.download("distilbert.zip")